In [98]:
import pandas as pd
df = pd.read_csv('wuzzuf_jobs.csv')
df

,Track,Job Title,Company,Location,Job Type,Job Experience,Job URL
0,Data Engineer,Associate System Engineer “Associate Data Mana...,Target Integrated Systems -,"Cairo, Egypt",Full Time / On-site,0 - 3 Yrs of Exp,https://wuzzuf.net/jobs/p/ubsrja3n94m1-associa...
1,Data Engineer,Senior Data Engineer,Vertex Technologies -,"Cairo, Egypt",Full Time / On-site,5+ Yrs of Exp,https://wuzzuf.net/jobs/p/3x6prdwpt2tz-senior-...
2,Data Engineer,Data Center Facility Electrical/ Mechanical En...,GPX Egypt -,"New Cairo, Cairo, Egypt",Full Time / On-site,3 - 5 Yrs of Exp,https://wuzzuf.net/jobs/p/39xm37penfcr-data-ce...
3,Data Engineer,Data Engineer (Infrastructure & Operations),MEAHCO - Saudi German Health -,"Katameya, Cairo, Egypt",Full Time / On-site,5 - 8 Yrs of Exp,https://wuzzuf.net/jobs/p/mhqqwhx31ryd-data-en...
4,Data Engineer,Sr. Data Engineer - Spark,Techsa -,"Cairo, Egypt",Full Time / Remote,7+ Yrs of Exp,https://wuzzuf.net/jobs/p/4w6f7tucldaf-sr-data...
...,...,...,...,...,...,...,...
927,Machine Learning Engineer,Junior System Administrator,SHM Industries -,"Cairo, Egypt",Full Time / On-site,1 - 2 Yrs of Exp,https://wuzzuf.net/jobs/p/esh3hhmwpxnt-junior-...
928,Machine Learning Engineer,Junior GIS Technician,Citylogix ME -,"Katameya, Cairo, Egypt",Full Time / On-site,0 - 3 Yrs of Exp,https://wuzzuf.net/jobs/p/9brwrgjkuyjv-junior-...
929,Machine Learning Engineer,Maintenance & Technical Support Specialist,Infinity tecs -,"Cairo, Egypt",Internship / On-site,NaN,https://wuzzuf.net/internship/okjfuagtatr5-mai...
930,Machine Learning Engineer,Python Developer (CAD Automation),StoryGold -,"San Francisco, United States",Full Time / Remote,0 - 2 Yrs of Exp,https://wuzzuf.net/jobs/p/hzdsg3stdbez-python-...


### 1. Duplicate job urls across different tracks merged into a new column

In [99]:
df['Job URL'].nunique()

670

In [100]:
original_columns = df.columns.tolist()

In [101]:
def merge_tracks(tracks):
    return ', '.join(tracks.drop_duplicates())

tracks_per_job = df.groupby('Job URL')['Track'].apply(merge_tracks).reset_index()

In [102]:
df = df.drop_duplicates(subset='Job URL', keep='first')

In [103]:
df = df.drop(columns='Track')
df = df.merge(tracks_per_job, on='Job URL', how='left')

In [104]:
df = df[original_columns]

In [105]:
len(df)

670

In [106]:
df['Job URL'].duplicated().sum()

np.int64(0)

### 2. Standardize Column Names and Clean White Space Inside Columns

In [107]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
df

,track,job_title,company,location,job_type,job_experience,job_url
0,Data Engineer,Associate System Engineer “Associate Data Mana...,Target Integrated Systems -,"Cairo, Egypt",Full Time / On-site,0 - 3 Yrs of Exp,https://wuzzuf.net/jobs/p/ubsrja3n94m1-associa...
1,"Data Engineer, Data Science, AI Developer",Senior Data Engineer,Vertex Technologies -,"Cairo, Egypt",Full Time / On-site,5+ Yrs of Exp,https://wuzzuf.net/jobs/p/3x6prdwpt2tz-senior-...
2,Data Engineer,Data Center Facility Electrical/ Mechanical En...,GPX Egypt -,"New Cairo, Cairo, Egypt",Full Time / On-site,3 - 5 Yrs of Exp,https://wuzzuf.net/jobs/p/39xm37penfcr-data-ce...
3,"Data Engineer, AI Developer",Data Engineer (Infrastructure & Operations),MEAHCO - Saudi German Health -,"Katameya, Cairo, Egypt",Full Time / On-site,5 - 8 Yrs of Exp,https://wuzzuf.net/jobs/p/mhqqwhx31ryd-data-en...
4,"Data Engineer, Data Science, AI Developer",Sr. Data Engineer - Spark,Techsa -,"Cairo, Egypt",Full Time / Remote,7+ Yrs of Exp,https://wuzzuf.net/jobs/p/4w6f7tucldaf-sr-data...
...,...,...,...,...,...,...,...
665,Machine Learning Engineer,Electrical Engineer,scada -,"New Cairo, Cairo, Egypt",Full Time / Hybrid,0 - 3 Yrs of Exp,https://wuzzuf.net/jobs/p/jfrgsx9fqgea-electri...
666,Machine Learning Engineer,Senior Electrical Maintenance Engineer,Confidential -,"6th of October, Giza, Egypt",Full Time / On-site,3 - 5 Yrs of Exp,https://wuzzuf.net/jobs/p/ugm8zuthwgbs-senior-...
667,Machine Learning Engineer,Mechatronics Maintenance Engineer - Pharma,APEX Pharma -,"Badr City, Cairo, Egypt",Full Time / On-site,5+ Yrs of Exp,https://wuzzuf.net/jobs/p/mzsoupud0vur-mechatr...
668,Machine Learning Engineer,Junior System Administrator,SHM Industries -,"Cairo, Egypt",Full Time / On-site,1 - 2 Yrs of Exp,https://wuzzuf.net/jobs/p/esh3hhmwpxnt-junior-...


In [108]:
for col in df.columns:
    df[col] = df[col].str.strip()


### 3. Split Employment Type and Work Mode

In [109]:
df['job_type'].value_counts(dropna=False)

job_type
Full Time / On-site                                     520
Full Time / Hybrid                                       58
Full Time / Remote                                       43
Freelance / Project / Remote                              7
Internship / Hybrid                                       6
Internship / On-site                                      6
Part Time / Remote                                        5
Full Time / Shift Based / On-site                         3
Full Time / Part Time / Freelance / Project / Hybrid      3
Full Time / Part Time / Hybrid                            2
Freelance / Project / Hybrid                              2
Full Time / Part Time / Remote                            2
Full Time / Freelance / Project / Hybrid                  2
Internship / Remote                                       2
Full Time / Part Time / On-site                           2
Part Time / Hybrid                                        1
Freelance / Project / On-site  

In [110]:
df['job_type'] = df['job_type'].str.replace('Freelance / Project', 'Freelance-Project', regex=False)

parts = df['job_type'].str.split(' / ')

df['work_mode'] = parts.str[-1]
df['employment_type'] = parts.str[:-1].str.join(', ')

df['employment_type'] = df['employment_type'].str.replace('Freelance-Project', 'Freelance / Project', regex=False)

### 4. Final Conversion to CSV file

In [111]:
df.to_csv('wuzzuf_jobs_cleaned.csv', index=False)